In [1]:
from utils import *
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

## Linear Regression

Simple population-level linear regression model. The country is not considered.

In [2]:
X, Y, C = load_height_data()
model = LinearRegression().fit(X, Y)
y_pred = model.predict(X)
print("Loss:", np.mean((y_pred - Y)**2))

Loss: 45.53098


Next, we try learning a separate linear regression model for each country. Notice the loss is much better.

In [3]:
X, Y, C = load_height_data()
model = ContextLinearRegression().fit(X, Y, C)
y_pred = model.predict(X, C)
print("Loss:", np.mean((y_pred - Y)**2))

Loss: 23.753902446756058


## Multilayer Perceptron
The MLP without country as input performs about equally well as learning a separate linear regression model for each country.

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = "cpu"
print(f"Using {device} device")

Using mps device


In [5]:
model = NeuralNetwork(dim_in=3, dim_out=1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [6]:
full_dataset = HeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=500)
test_dataloader = DataLoader(test_data, batch_size=500)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test(test_dataloader, model, loss_fn, device)
print("Done!")

Epoch 1
-------------------------------
loss: 413.282593  [  500/168000]
loss: 37.099915  [50500/168000]
loss: 31.862421  [100500/168000]
loss: 28.095755  [150500/168000]
Test Error: 
 Avg loss: 28.954300 

Epoch 2
-------------------------------
loss: 29.107752  [  500/168000]
loss: 31.142996  [50500/168000]
loss: 26.963802  [100500/168000]
loss: 26.597490  [150500/168000]
Test Error: 
 Avg loss: 26.451828 

Epoch 3
-------------------------------
loss: 26.851067  [  500/168000]
loss: 29.774364  [50500/168000]
loss: 25.614752  [100500/168000]
loss: 25.680170  [150500/168000]
Test Error: 
 Avg loss: 25.169939 

Epoch 4
-------------------------------
loss: 25.279766  [  500/168000]
loss: 28.691629  [50500/168000]
loss: 24.903036  [100500/168000]
loss: 25.448666  [150500/168000]
Test Error: 
 Avg loss: 24.409281 

Epoch 5
-------------------------------
loss: 24.387156  [  500/168000]
loss: 28.001423  [50500/168000]
loss: 24.482073  [100500/168000]
loss: 24.672258  [150500/168000]
Test 

## Context-Sensitive Network
The task is one-hot-encoded and appended to the input.